# Debate Analysis Notebook

Loads the most recent `DebateResult` JSON from `results/debates/` and produces the visualizations and tables that ship with the README submission.

**Run with:** `uv run jupyter notebook notebooks/analysis.ipynb`

**Required state:** at least one debate must have been run (`uv run python -m debate` → option 1) so a result JSON exists. If `results/debates/` is empty this notebook will tell you and exit cleanly.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

from debate.shared.schemas import DebateResult

RESULTS_DIR = Path('../results/debates')
ASSETS_DIR = Path('../assets')
ASSETS_DIR.mkdir(exist_ok=True)

files = sorted(RESULTS_DIR.glob('debate_*.json')) if RESULTS_DIR.exists() else []
if not files:
    print('No debates found in', RESULTS_DIR.resolve())
    print('Run `uv run python -m debate` and choose option 1 first.')
    raise SystemExit
latest = files[-1]
print('Analyzing', latest.name)
result = DebateResult.model_validate(json.loads(latest.read_text(encoding='utf-8')))

## 1. Verdict summary

In [ ]:
v = result.verdict
print(f'Topic: {result.topic}')
print(f'Winner: {v.winner.upper()}')
print(f'Dogs total: {v.dogs_total} | Cats total: {v.cats_total} | Margin: {v.margin}')
print()
print('Written rationale:')
print(v.written_rationale)

## 2. Total score comparison

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Dogs', 'Cats'], [v.dogs_total, v.cats_total], color=['#3b6e8f', '#bf6f4a'])
ax.set_ylabel('Total score')
ax.set_title('Total scores — winner: ' + v.winner.upper())
for i, total in enumerate([v.dogs_total, v.cats_total]):
    ax.text(i, total + 0.5, str(total), ha='center')
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'total_scores.png', dpi=150)
plt.show()

## 3. Score breakdown per dimension (stacked bar)

In [ ]:
dimensions = ['structure', 'logos', 'pathos', 'ethos', 'clash']
totals = {side: {d: 0 for d in dimensions} for side in ('dogs', 'cats')}
for s in result.scores:
    for d in dimensions:
        totals[s.side][d] += getattr(s, d)

fig, ax = plt.subplots(figsize=(7, 4))
bottoms = {'dogs': 0, 'cats': 0}
colors = ['#5b8db8', '#7fa8c9', '#a3c3da', '#c7debc', '#e9b984']
for d, color in zip(dimensions, colors):
    vals = [totals['dogs'][d], totals['cats'][d]]
    ax.bar(['Dogs', 'Cats'], vals, bottom=[bottoms['dogs'], bottoms['cats']],
           label=d, color=color)
    bottoms['dogs'] += vals[0]
    bottoms['cats'] += vals[1]
ax.set_title('Score breakdown by rhetorical dimension')
ax.legend(loc='upper right')
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'score_breakdown.png', dpi=150)
plt.show()

## 4. Per-round clash evolution

In [ ]:
rounds = sorted({s.ping_round for s in result.scores})
dogs_clash = [next((s.clash for s in result.scores if s.ping_round == r and s.side == 'dogs'), 0) for r in rounds]
cats_clash = [next((s.clash for s in result.scores if s.ping_round == r and s.side == 'cats'), 0) for r in rounds]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(rounds, dogs_clash, marker='o', label='Dogs', color='#3b6e8f')
ax.plot(rounds, cats_clash, marker='s', label='Cats', color='#bf6f4a')
ax.set_xlabel('Round')
ax.set_ylabel('Clash score (0–3)')
ax.set_title('Clash engagement per round')
ax.set_ylim(-0.2, 3.2)
ax.legend()
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'clash_per_round.png', dpi=150)
plt.show()

## 5. Cost breakdown

Per the cost formula:

$$\text{cost}(m) = \frac{p_{\text{in}}}{10^6} \cdot t_{\text{in}} \;+\; \frac{p_{\text{out}}}{10^6} \cdot t_{\text{out}} \;+\; 1.25 \cdot \frac{p_{\text{in}}}{10^6} \cdot t_{\text{cache-write}} \;+\; 0.10 \cdot \frac{p_{\text{in}}}{10^6} \cdot t_{\text{cache-read}}$$

where $p_{\text{in}}, p_{\text{out}}$ are the per-million-token prices for model $m$ and $t_*$ are token counts.

In [ ]:
report = result.cost_report
rows = []
for model, stats in (report.get('by_model') or {}).items():
    rows.append((model, stats.get('input_tokens', 0), stats.get('output_tokens', 0),
                 stats.get('cache_read_tokens', 0), stats.get('cost_usd', 0)))

print(f"{'Model':<35} {'In':>10} {'Out':>10} {'Cached':>10} {'Cost (USD)':>12}")
print('-' * 80)
for model, tin, tout, cached, cost in rows:
    print(f'{model:<35} {tin:>10} {tout:>10} {cached:>10} {cost:>12.6f}')
print('-' * 80)
print(f"Total: ${report.get('total_usd', 0):.6f}  |  cache read share: {report.get('cache_read_pct', 0):.1f}%")

## 6. Conclusion

Hand-edit this cell after running the notebook against a real debate transcript: which side won, by what margin, which rhetorical dimension drove the result, and whether the cost stayed inside the configured `budget_usd` from `config/setup.json`.